In [18]:
import pandas as pd
import scipy
from scipy import stats
import numpy as np

from eval_functions import correctness_score, hallucination_score, hallucination_score_noramlized

vision_base_eval = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-hallucination-correctness-base.csv")

#### Scoring criteria: Correctness

- Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
- Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
- Score 2 — The generated answer is correct and captures the key facts from the reference answer.

#### Scoring criteria: Hallucination

- Score 0 — The answer contains coaching advice that directly contradicts the retrieved context, or invents training recommendations with no basis in the context. This is the worst outcome.
- Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
- Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

In [3]:
vision_base_eval.head()

,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,2497560a-f99e-4d90-9374-928b4f5158be,"{""user_query"": ""are there any issues with my f...","{""answer"": ""The clearest visible issues are th...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the effort here—let’s dial a ...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,147.685615,20358,0.055014,1.0,1.0
1,289d6134-9559-4328-aa64-eac2084de0a0,"{""user_query"": ""How's my bench?""}","{""answer"": ""Slight movement in the feet. Wrist...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the outdoor grind—let’s dial ...","{""inputs"": {""user_query"": ""How's my bench?""}, ...",success,NaN,53.165919,10482,0.037487,0.5,2.0
2,308047c5-2ad2-4dfb-8c19-b983543ab83d,"{""user_query"": ""Can you check my form?""}","{""answer"": ""The biggest issue I can see is you...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Absolutely—thanks for the pics! He...","{""inputs"": {""user_query"": ""Can you check my fo...",success,NaN,62.943568,10241,0.033891,1.0,1.0
3,3273e894-50bb-46c3-a9e8-2d2831f40376,"{""user_query"": ""are there any issues with my f...","{""answer"": ""Your back looks completely flat on...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Awesome work getting full-range be...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,64.178148,15358,0.044209,0.5,1.0
4,58cbb032-850b-43f8-954b-f8ac3b9367f9,"{""user_query"": ""Are there any issues with my f...","{""answer"": ""The forearms do not appear vertica...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Awesome job getting after it. I wa...","{""inputs"": {""user_query"": ""Are there any issue...",success,NaN,54.787645,10332,0.034054,1.0,2.0


# Vision Base Eval: 10 Samples

In [4]:
# Find the standard deviation of the correctness and hallucination scores
vision_base_eval.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,10.0,0.0,10.000000,10.000000,10.000000,10.000000,10.000000
mean,1.0,NaN,67.350450,12951.800000,0.039158,0.750000,1.500000
std,0.0,NaN,32.852826,4344.870329,0.008369,0.263523,0.527046
min,1.0,NaN,42.945023,7923.000000,0.029911,0.500000,1.000000
25%,1.0,NaN,48.234488,10263.750000,0.033932,0.500000,1.000000
50%,1.0,NaN,53.976782,10423.000000,0.036674,0.750000,1.500000
75%,1.0,NaN,63.869503,15190.000000,0.043058,1.000000,2.000000
max,1.0,NaN,147.685615,20358.000000,0.055014,1.000000,2.000000


In [5]:

base_10_hallucination = hallucination_score(vision_base_eval, df_name="vision_base_eval_10")
hallucination_score_mean = vision_base_eval["hallucination"].mean()
print(f"Hallcination score overall {hallucination_score_mean}")

print(" ")
base_10_correctness = correctness_score(vision_base_eval, df_name="vision_base_eval_10")
correctness_score_mean = vision_base_eval["correctness"].mean() * 2 # normalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")


95% CI Hallucination Low - vision_base_eval_10: 1.2
95% CI Hallucination High - vision_base_eval_10: 1.8
Standard Error - Hallucination - vision_base_eval_10: 0.1554
hallucination_MOE - vision_base_eval_10: 0.30
Hallcination score overall 1.5
 
95% CI Correctness Low - vision_base_eval_10: 1.2
95% CI Correctness High - vision_base_eval_10: 1.8
Standard Error - Correctness - vision_base_eval_10: 0.1574
Correctness MOE - vision_base_eval_10: 0.30
Correctness score overall: 1.5


#### Findings

- Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
- Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

Hallucination scores range between 1.2 and 1.8. This means they drift between containing some hallucinations (unsupported recommendations) and none. However, with n=10, it is impossible to say where the system truly sits between these scores.

#### Next steps

What happens if n=20, or n=30? Apply formula:
- new_MOE = old_MOE / sqrt(n2/n1)

In [6]:
hallucination_20_MOE = 0.3 / (np.sqrt(20/10))
print(f"MOE - 20 samples: {hallucination_20_MOE:.2f}")

hallucination_30_MOE = 0.3 / (np.sqrt(30/10))
print(f"MOE - 30 samples: {hallucination_30_MOE:.2f}")

MOE - 20 samples: 0.21
MOE - 30 samples: 0.17


# Vision Base Eval 20: test_1

In [7]:
vision_base_eval_20 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-20samples.csv")
vision_base_eval_20.head()

,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,1d422e61-d9b9-461d-9514-d167322c325a,"{""user_query"": ""How's my form?""}","{""answer"": ""Wrists are extended back under the...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Awesome work getting after it. Her...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,42.583386,10285,0.031829,0.5,1.0
1,2497560a-f99e-4d90-9374-928b4f5158be,"{""user_query"": ""are there any issues with my f...","{""answer"": ""The clearest visible issues are th...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the effort here—let’s dial a ...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,42.086563,19562,0.045299,1.0,1.0
2,289d6134-9559-4328-aa64-eac2084de0a0,"{""user_query"": ""How's my bench?""}","{""answer"": ""Slight movement in the feet. Wrist...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love the backyard grind! Thanks fo...","{""inputs"": {""user_query"": ""How's my bench?""}, ...",success,NaN,46.255542,10157,0.035699,0.5,1.0
3,308047c5-2ad2-4dfb-8c19-b983543ab83d,"{""user_query"": ""Can you check my form?""}","{""answer"": ""The biggest issue I can see is you...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love that you filmed your bench—le...","{""inputs"": {""user_query"": ""Can you check my fo...",success,NaN,69.200048,10121,0.030731,1.0,2.0
4,3273e894-50bb-46c3-a9e8-2d2831f40376,"{""user_query"": ""are there any issues with my f...","{""answer"": ""Your back looks completely flat on...",vision-faithfulness-rag-v7-structured-output-4...,1,"{""answer"": ""Love that you filmed this—great st...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,45.183090,14805,0.039676,1.0,1.0


In [8]:
vision_base_eval_20.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.00000,20.000000
mean,1.0,NaN,47.563381,13743.700000,0.037699,0.65000,1.600000
std,0.0,NaN,13.011740,4295.202343,0.005575,0.28562,0.502625
min,1.0,NaN,37.114877,8450.000000,0.028813,0.00000,1.000000
25%,1.0,NaN,41.421420,10228.500000,0.033708,0.50000,1.000000
50%,1.0,NaN,44.581522,10777.000000,0.036770,0.50000,2.000000
75%,1.0,NaN,47.141303,18730.000000,0.041026,1.00000,2.000000
max,1.0,NaN,94.627459,19562.000000,0.047601,1.00000,2.000000


In [9]:
base_20_hallucination = hallucination_score(vision_base_eval_20, df_name="vision_base_eval_20_test1")
hallucination_score_mean = vision_base_eval_20["hallucination"].mean()
print(f"Hallcination score overall {hallucination_score_mean}")


print(" ")
base_20_correctness = correctness_score(vision_base_eval_20, df_name="vision_base_eval_20_test1")
correctness_score_mean = vision_base_eval_20["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

95% CI Hallucination Low - vision_base_eval_20_test1: 1.4
95% CI Hallucination High - vision_base_eval_20_test1: 1.8
Standard Error - Hallucination - vision_base_eval_20_test1: 0.1081
hallucination_MOE - vision_base_eval_20_test1: 0.20
Hallcination score overall 1.6
 
95% CI Correctness Low - vision_base_eval_20_test1: 1.05
95% CI Correctness High - vision_base_eval_20_test1: 1.55
Standard Error - Correctness - vision_base_eval_20_test1: 0.1267
Correctness MOE - vision_base_eval_20_test1: 0.25
Correctness score overall: 1.3


# Vision Eval 20: test_2
- correctness judge prompt update, normralized hallucination score

In [10]:
vision_eval_20_test2 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-eval_ correctness prompt update, normralized hallucination score.csv")
vision_eval_20_test2.head()

,id,inputs,reference_outputs,session_name,repetition,outputs,run,status,error,latency,tokens,total_cost,correctness,hallucination
0,1d422e61-d9b9-461d-9514-d167322c325a,"{""user_query"": ""How's my form?""}","{""answer"": ""Wrists are extended back under the...",vision-faithfulness-rag-eval: correctness prom...,1,"{""answer"": ""Love the effort here! You’re clear...","{""inputs"": {""user_query"": ""How's my form?""}, ""...",success,NaN,56.941229,10514,0.035764,0.5,0.5
1,2497560a-f99e-4d90-9374-928b4f5158be,"{""user_query"": ""are there any issues with my f...","{""answer"": ""The clearest visible issues are th...",vision-faithfulness-rag-eval: correctness prom...,1,"{""answer"": ""Love the effort here—strong reps! ...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,67.406299,19507,0.046994,1.0,0.5
2,289d6134-9559-4328-aa64-eac2084de0a0,"{""user_query"": ""How's my bench?""}","{""answer"": ""Slight movement in the feet. Wrist...",vision-faithfulness-rag-eval: correctness prom...,1,"{""answer"": ""Love the outdoor grind! Here’s a q...","{""inputs"": {""user_query"": ""How's my bench?""}, ...",success,NaN,37.357729,10281,0.035268,0.5,0.5
3,308047c5-2ad2-4dfb-8c19-b983543ab83d,"{""user_query"": ""Can you check my form?""}","{""answer"": ""The biggest issue I can see is you...",vision-faithfulness-rag-eval: correctness prom...,1,"{""answer"": ""Love it—thanks for the pics. Let’s...","{""inputs"": {""user_query"": ""Can you check my fo...",success,NaN,57.666777,10830,0.037820,1.0,1.0
4,3273e894-50bb-46c3-a9e8-2d2831f40376,"{""user_query"": ""are there any issues with my f...","{""answer"": ""Your back looks completely flat on...",vision-faithfulness-rag-eval: correctness prom...,1,"{""answer"": ""Awesome work getting video from th...","{""inputs"": {""user_query"": ""are there any issue...",success,NaN,50.512474,15352,0.044228,1.0,0.5


In [11]:
vision_eval_20_test2.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.00000,20.000000
mean,1.0,NaN,57.631318,13960.550000,0.039849,0.65000,0.800000
std,0.0,NaN,19.424917,4390.688706,0.005966,0.28562,0.251312
min,1.0,NaN,37.357729,8704.000000,0.029735,0.00000,0.500000
25%,1.0,NaN,49.284036,10455.750000,0.036044,0.50000,0.500000
50%,1.0,NaN,53.477690,10766.500000,0.037972,0.50000,1.000000
75%,1.0,NaN,59.696380,19285.500000,0.044887,1.00000,1.000000
max,1.0,NaN,132.665246,19721.000000,0.049007,1.00000,1.000000


In [12]:
print("Hallucination")
vision_20_hallucination_test2 = hallucination_score_noramlized(vision_eval_20_test2, df_name="vision_eval_20_test2")
hallucination_score_mean = vision_eval_20_test2["hallucination"].mean() * 2 # denormalize the overall score
print(f"Hallcination score overall {hallucination_score_mean}")

print(" ")

print("Correctness")
vision_20_correctness_test2 = correctness_score(vision_eval_20_test2, df_name="vision_eval_20_test2")
correctness_score_mean = vision_eval_20_test2["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

Hallucination
95% CI Hallucination Low - vision_eval_20_test2: 1.35
95% CI Hallucination High - vision_eval_20_test2: 1.8
Standard Error - Hallucination - vision_eval_20_test2: 0.1099
hallucination_MOE - vision_eval_20_test2: 0.22
Hallcination score overall 1.6
 
Correctness
95% CI Correctness Low - vision_eval_20_test2: 1.05
95% CI Correctness High - vision_eval_20_test2: 1.55
Standard Error - Correctness - vision_eval_20_test2: 0.1252
Correctness MOE - vision_eval_20_test2: 0.25
Correctness score overall: 1.3


# vision-faithfulness-rag-updated-system-prompt-test3
- updated system prompt eval, 20 samples

In [17]:
vision_eval_test3 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-rag-updated-system-prompt-test3.csv")

print("Hallucination")
vision_20_hallucination_test3 = hallucination_score_noramlized(vision_eval_test3, df_name="vision_eval_test3")
hallucination_score_mean = vision_eval_test3["hallucination"].mean() * 2 # denormalize the overall score
print(f"Hallcination score overall {hallucination_score_mean}")

print(" ")

print("Correctness")
vision_20_correctness_test3 = correctness_score(vision_eval_test3, df_name="vision_eval_test3")
correctness_score_mean = vision_eval_test3["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

Hallucination
95% CI Hallucination Low - vision_eval_test3: 1.35
95% CI Hallucination High - vision_eval_test3: 1.8
Standard Error - Hallucination - vision_eval_test3: 0.1101
hallucination_MOE - vision_eval_test3: 0.22
Hallcination score overall 1.6
 
Correctness
95% CI Correctness Low - vision_eval_test3: 1.3
95% CI Correctness High - vision_eval_test3: 1.7
Standard Error - Correctness - vision_eval_test3: 0.1137
Correctness MOE - vision_eval_test3: 0.20
Correctness score overall: 1.5


# Results: 

### Base eval - 10 samples: 
**Hallucination**
- 95% CI Hallucination Low - vision_base_eval_10: 1.2
- 95% CI Hallucination High - vision_base_eval_10: 1.8
- Standard Error - Hallucination - vision_base_eval_10: 0.1554
- hallucination_MOE - vision_base_eval_10: 0.30
- Hallcination score overall 1.5

**Correctness** 
- 95% CI Correctness Low - vision_base_eval_10: 1.2
- 95% CI Correctness High - vision_base_eval_10: 1.8
- Standard Error - Correctness - vision_base_eval_10: 0.1574
- Correctness MOE - vision_base_eval_10: 0.30
- Correctness score overall: 1.5

### Base eval - 20 samples
**Hallucination**
- 95% CI Hallucination Low - vision_base_eval_20_test1: 1.4
- 95% CI Hallucination High - vision_base_eval_20_test1: 1.8
- Standard Error - Hallucination - vision_base_eval_20_test1: 0.1081
- Hallucination MOE - vision_base_eval_20_test1: 0.20

 **Correctness** 
- 95% CI Correctness Low - vision_base_eval_20_test1: 1.05
- 95% CI Correctness High - vision_base_eval_20_test1: 1.55
- Standard Error - Correctness - vision_base_eval_20_test1: 0.1267
- Correctness MOE - vision_base_eval_20_test1: 0.25
- Correctness score overall: 1.3


### Vision Eval - test_2 (correctness prompt update):

**Hallucination**
- 95% CI Hallucination Low - vision_eval_20_test2: 1.35
- 95% CI Hallucination High - vision_eval_20_test2: 1.8
- Standard Error - Hallucination - vision_eval_20_test2: 0.1099
- hallucination_MOE - vision_eval_20_test2: 0.22
- Hallcination score overall 1.6
 
**Correctness**
- 95% CI Correctness Low - vision_eval_20_test2: 1.05
- 95% CI Correctness High - vision_eval_20_test2: 1.55
- Standard Error - Correctness - vision_eval_20_test2: 0.1252
- Correctness MOE - vision_eval_20_test2: 0.25
- Correctness score overall: 1.3


### Vision Eval - test_3 (system prompt update):

**Hallucination**
95% CI Hallucination Low - vision_eval_20_test2: 1.35
95% CI Hallucination High - vision_eval_20_test2: 1.8
Standard Error - Hallucination - vision_eval_20_test2: 0.1101
hallucination_MOE - vision_eval_20_test2: 0.22
Hallcination score overall 1.6
 
**Correctness**
95% CI Correctness Low - vision_eval_20_test2: 1.3
95% CI Correctness High - vision_eval_20_test2: 1.7
Standard Error - Correctness - vision_eval_20_test2: 0.1137
Correctness MOE - vision_eval_20_test2: 0.20
Correctness score overall: 1.5

### Outcome

20 samples reduced the margin of error for hallucination from 0.3 to 0.2. For correctness, it fell from 0.3 to 0.25.


## Conduct t-test
- Null hypothesis (H0):

- H1: 


In [19]:
baseline_scores = vision_base_eval_20["correctness"]
new_scores = vision_eval_test3["correctness"]

res_correctness = scipy.stats.ttest_ind(baseline_scores, new_scores)
res_correctness

TtestResult(statistic=np.float64(-1.1649647450214349), pvalue=np.float64(0.2512944934388181), df=np.float64(38.0))

In [20]:
baseline_scores = vision_base_eval_20["hallucination"]
new_scores = vision_eval_test3["hallucination"]

res_hallucination = scipy.stats.ttest_ind(baseline_scores, new_scores)
res_hallucination

TtestResult(statistic=np.float64(6.366579406033772), pvalue=np.float64(1.7906315879054875e-07), df=np.float64(38.0))

In [21]:
baseline_scores = vision_eval_20_test2["correctness"]
new_scores = vision_eval_test3["correctness"]

res_correctness_2 = scipy.stats.ttest_ind(baseline_scores, new_scores)
res_correctness_2

TtestResult(statistic=np.float64(-1.1649647450214349), pvalue=np.float64(0.2512944934388181), df=np.float64(38.0))

In [22]:
baseline_scores = vision_eval_20_test2["hallucination"]
new_scores = vision_eval_test3["hallucination"]

res_hallucination_2 = scipy.stats.ttest_ind(baseline_scores, new_scores)
res_hallucination_2

TtestResult(statistic=np.float64(0.0), pvalue=np.float64(1.0), df=np.float64(38.0))

# Model Swap Comparison: gpt-5 → gpt-5.4 (Response Generator)

In [26]:
vision_eval_20_test_4 = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/llm-evals/vision-faithfulness-gpt-5.4-response_generator.csv")
vision_eval_20_test_4.describe()

,repetition,error,latency,tokens,total_cost,correctness,hallucination
count,20.0,0.0,20.000000,20.000000,20.000000,20.000000,20.000000
mean,1.0,NaN,24.849006,14675.300000,0.041166,0.725000,0.750000
std,0.0,NaN,4.092166,6002.343271,0.014363,0.255209,0.256495
min,1.0,NaN,20.083819,6416.000000,0.020617,0.500000,0.500000
25%,1.0,NaN,22.820954,9829.750000,0.029325,0.500000,0.500000
50%,1.0,NaN,24.168265,11795.000000,0.033617,0.500000,0.750000
75%,1.0,NaN,25.836488,21842.500000,0.057639,1.000000,1.000000
max,1.0,NaN,37.365576,22000.000000,0.059112,1.000000,1.000000


In [27]:
print("Hallucination (test 4 - gpt-5.4)")
vision_20_hallucination_test_4 = hallucination_score_noramlized(vision_eval_20_test_4, df_name="vision_eval_test_4")
hallucination_score_mean = vision_eval_20_test_4["hallucination"].mean() * 2 # denormalize the overall score
print(f"Hallcination score overall {hallucination_score_mean}")

print(" ")

print("Correctnes (test4 - gpt 5.4")
vision_20_correctness_test_4 = correctness_score(vision_eval_20_test_4, df_name="vision_eval_test_4")
correctness_score_mean = vision_eval_20_test_4["correctness"].mean() * 2 # denormalize the overall score
print(f"Correctness score overall: {correctness_score_mean}")

Hallucination (test 4 - gpt-5.4)
95% CI Hallucination Low - vision_eval_test_4: 1.3
95% CI Hallucination High - vision_eval_test_4: 1.7
Standard Error - Hallucination - vision_eval_test_4: 0.1130
hallucination_MOE - vision_eval_test_4: 0.20
Hallcination score overall 1.5
 
Correctnes (test4 - gpt 5.4
95% CI Correctness Low - vision_eval_test_4: 1.25
95% CI Correctness High - vision_eval_test_4: 1.65
Standard Error - Correctness - vision_eval_test_4: 0.1115
Correctness MOE - vision_eval_test_4: 0.20
Correctness score overall: 1.45


# Fitness Form Coach — Optimization Results

## 1. Model Swap: gpt-4o-mini → gpt-5.4-nano

### Nodes Changed
- Router (query classification)
- Chat memory (conversational responses)
- Classifier (exercise classification)

### Latency Comparison

| Query | Before (gpt-4o-mini) | After (gpt-5.4-nano) | Change |
|---|---|---|---|
| "thanks for your help" (chat memory) | 8.21s | 5.32s | -35% |
| "can you tell me again..." (follow-up) | 15.95s | 8.08s | -49% |
| Bench press text query (vector_db → response_gen) | 43.63s | 42.02s | -4% |
| Video query (full pipeline) | 103.80s | 58.11s | -44% |

### Cost Comparison

| Query | Before cost | After cost | Change |
|---|---|---|---|
| "thanks for your help" (chat memory) | $0.0452 | $0.0012 | -97% |
| "can you tell me again..." (follow-up) | $0.0474 | $0.0040 | -92% |
| Bench press text query | $0.0350 | $0.0301 | -14% |
| Video query (full pipeline) | $0.0500 | $0.0452 | -10% |

### Key Takeaways
- Chat memory and follow-up queries saw the biggest improvements: ~35-49% latency reduction and ~90-97% cost reduction.
- Video query latency nearly halved (103.8s → 58.1s), driven by the classifier switch to nano.
- Text queries routed through vector_db → response_generator barely changed (~4%) because the bottleneck is the response_generator LLM — not yet optimized at this stage.

---

## 2. Memory Summarization

### Change
- Removed `RunnableWithMessageHistory` from `response_generator`
- Manually manage history: save summarized response (via gpt-5.4-nano) instead of full video analysis
- User still sees the full response; history stores a ~150-word summary
- Summarization prompt extracts: exercise, main issues, priority fixes, uncertainties, and a concise coaching summary

### Results (same session: video → text query → follow-up → thanks)

| Query | Before (full history) | After (summarized) | Change |
|---|---|---|---|
| "thanks for your help" tokens | 18,300 | 328 | -98% |
| "thanks for your help" latency | 4.52s | 1.74s | -62% |
| Text follow-up tokens | 18,100 | 2,863 | -84% |
| Text follow-up latency | — | 4.48s | — |
| "can you tell me again..." tokens | — | 1,372 | — |

### Key Takeaways
- Token usage on follow-up messages dropped 84-98% because history no longer contains the full video analysis.
- Response quality on follow-ups is unchanged — the summary retains enough context for the model to answer questions about tempo, grip, form cues, etc.
- The summarization adds a one-time nano LLM call after each video analysis, but every subsequent message in the session benefits from the reduced history size.

---

## 3. Model Swap: GPT-5 → GPT-5.4 (Response Generator)

### Node Changed
- Response generator (video analysis)

### Quality Comparison (LangSmith evals, n=20)

| Metric | GPT-5 (baseline) | GPT-5.4 | Change |
|---|---|---|---|
| Correctness | 0.75 | 0.72 | -0.03 (not significant) |
| Hallucination | 0.80 | 0.75 | +0.05 (not significant) |

### Latency Comparison

| Metric | GPT-5 | GPT-5.4 | Change |
|---|---|---|---|
| P50 latency | 53.71s | 24.17s | -55% |
| P99 latency | 103.13s | 35.93s | -65% |

### Key Takeaways
- Correctness and hallucination scores are effectively unchanged — the -0.03 and +0.05 differences are well within the margin of error for 20 samples.
- P50 latency dropped 55%, P99 dropped 65%. The video analysis pipeline is now more than twice as fast.
- GPT-5.4 is a clear win: same quality, dramatically faster.

---

## Cumulative Impact

| Metric | Original | Final | Total Change |
|---|---|---|---|
| Video analysis latency (P50) | 103.80s | 24.17s | -77% |
| "thanks for your help" tokens | 18,300 | 328 | -98% |
| "thanks for your help" latency | 8.21s | 1.74s | -79% |
| "thanks for your help" cost | $0.0452 | $0.0001 | -99% |
| Follow-up query tokens | 18,100 | 1,372 | -92% |